Fanny BADOULES  
Maëlys HANOIRE  
Diane VERBECQ  


# Projet SDP

## Question 1

Formuler un programme d’optimisation linéaire qui calcule une explication de
type (1-1) de la comparaison x ≻ y si elle existe, et retourne un certificat de non-existence
dans le cas contraire. Impl´ementez cette formulation en utilisant un solveur d’optimisation.

In [33]:
from gurobipy import Model, GRB, quicksum

# Données de l'exemple x (Xavier) vs y (Yvonne)
courses = ["A","B","C","D","E","F","G"]

x = {"A":85, "B":81, "C":71, "D":69, "E":75, "F":81, "G":88}
y = {"A":81, "B":81, "C":75, "D":63, "E":67, "F":88, "G":95}

w = {"A":8, "B":7, "C":7, "D":6, "E":6, "F":5, "G":6}


In [34]:
# Contributions
delta = {k: w[k]*(x[k]-y[k]) for k in courses}

pros    = [k for k in courses if delta[k] > 0]
cons    = [k for k in courses if delta[k] < 0]
neutral = [k for k in courses if delta[k] == 0]

print("Δ:", delta)
print("pros(x,y):", pros)
print("cons(x,y):", cons)
print("neutral(x,y):", neutral)

# Ensemble T des paires admissibles (1-1)
T = [(p,c) for p in pros for c in cons if delta[p] + delta[c] > 0]
print("\nNombre de trade-offs admissibles |T| =", len(T))
print("T =", T)


Δ: {'A': 32, 'B': 0, 'C': -28, 'D': 36, 'E': 48, 'F': -35, 'G': -42}
pros(x,y): ['A', 'D', 'E']
cons(x,y): ['C', 'F', 'G']
neutral(x,y): ['B']

Nombre de trade-offs admissibles |T| = 6
T = [('A', 'C'), ('D', 'C'), ('D', 'F'), ('E', 'C'), ('E', 'F'), ('E', 'G')]


On calcule la contribution de chaque matière à la comparaison x>y, puis on classe les matières en favorables (pros), défavorables (cons) ou neutres.

On en déduit l’ensemble T des trade-offs (1-1) possibles, où une matière favorable compense une matière défavorable avec un gain global positif.

In [35]:
m = Model("Explanation_1-1")

# Variables binaires z[p,c] = 1 si on choisit le trade-off (p,c)
z = m.addVars(T, vtype=GRB.BINARY, name="z")

# Chaque critère "contre" doit être couvert exactement une fois
for c in cons:
    m.addConstr(quicksum(z[p,c] for p in pros if (p,c) in z) == 1, name=f"cover_{c}")

# Chaque critère "pour" utilisé au plus une fois (disjonction côté pros)
for p in pros:
    m.addConstr(quicksum(z[p,c] for c in cons if (p,c) in z) <= 1, name=f"use_{p}_at_most_once")

# Objectif : minimiser la longueur (nombre de paires)
m.setObjective(quicksum(z[p,c] for (p,c) in T), GRB.MINIMIZE)

# Mode silencieux
m.params.OutputFlag = 0

m.optimize()


On crée le modèle d’optimisation et des variables binaires qui indiquent quels trade-offs (1-1) sont sélectionnés pour expliquer
x>y.

Les contraintes imposent que chaque matière défavorable soit compensée exactement une fois, et l’objectif minimise le nombre de trade-offs utilisés pour obtenir l’explication la plus simple.

In [36]:
status = m.Status

if status == GRB.OPTIMAL:
    E = [(p,c) for (p,c) in T if z[p,c].X > 0.5]
    print("Explication (1-1) trouvée")
    print("Longueur l =", len(E))
    print("E =", E)

    covered_cons = sorted([c for (_,c) in E])
    print("Couverts (cons) =", covered_cons)

    print("\nInterprétation :")
    for p,c in E:
        print(f"- l'avantage en {p} (Δ={delta[p]}) compense le désavantage en {c} (Δ={delta[c]}), somme = {delta[p]+delta[c]}>0")

elif status == GRB.INFEASIBLE:
    print("Aucune explication (1-1) n'existe : modèle INFaisable.")


else:
    print("Statut solveur :", status)


Explication (1-1) trouvée
Longueur l = 3
E = [('A', 'C'), ('D', 'F'), ('E', 'G')]
Couverts (cons) = ['C', 'F', 'G']

Interprétation :
- l'avantage en A (Δ=32) compense le désavantage en C (Δ=-28), somme = 4>0
- l'avantage en D (Δ=36) compense le désavantage en F (Δ=-35), somme = 1>0
- l'avantage en E (Δ=48) compense le désavantage en G (Δ=-42), somme = 6>0


Le solveur trouve une explication (1-1) de longueur 3 : chaque matière défavorable à x>y (C, F, G) est compensée par une matière favorable (A, D, E) avec un bilan positif.
Concrètement, on justifie x>y par trois compensations : (A) bat (C), (D) bat (F) et (E) bat (G) (les sommes (4), (1) et (6) sont toutes (>0)).


## Question 2
Formuler un programme d’optimisation linéaire qui calcule une explication de type (1-m) de la comparaison x > y si elle existe, et retourne un certificat de non-existence dans le cas contraire. Implémentez cette formulation en utilisant un solveur d’optimisation.

Imports + données (tous les candidats)

In [37]:
from gurobipy import Model, GRB, quicksum
from itertools import combinations

courses = ["A","B","C","D","E","F","G"]
w = {"A":8, "B":7, "C":7, "D":6, "E":6, "F":5, "G":6}

scores = {
    "x": {"A":85, "B":81, "C":71, "D":69, "E":75, "F":81, "G":88},
    "y": {"A":81, "B":81, "C":75, "D":63, "E":67, "F":88, "G":95},
    "z": {"A":74, "B":89, "C":74, "D":81, "E":68, "F":84, "G":79},
    "t": {"A":74, "B":71, "C":84, "D":91, "E":77, "F":76, "G":73},
    "u": {"A":72, "B":75, "C":66, "D":85, "E":88, "F":66, "G":93},
    "v": {"A":71, "B":73, "C":63, "D":92, "E":76, "F":79, "G":93},
    "w": {"A":79, "B":69, "C":78, "D":76, "E":67, "F":84, "G":79},
    "w'":{"A":57, "B":76, "C":81, "D":76, "E":82, "F":86, "G":77},
}


Fonction utilitaire : calculer Δ, pros, cons

In [38]:
def compute_delta_pros_cons(name1, name2):
    x = scores[name1]
    y = scores[name2]
    delta = {k: w[k] * (x[k] - y[k]) for k in courses}
    pros = [k for k in courses if delta[k] > 0]
    cons = [k for k in courses if delta[k] < 0]
    neutral = [k for k in courses if delta[k] == 0]
    return delta, pros, cons, neutral

delta, pros, cons, neutral = compute_delta_pros_cons("w", "w'")
print("Δ:", delta)
print("pros:", pros)
print("cons:", cons)
print("neutral:", neutral)


Δ: {'A': 176, 'B': -49, 'C': -21, 'D': 0, 'E': -90, 'F': -10, 'G': 12}
pros: ['A', 'G']
cons: ['B', 'C', 'E', 'F']
neutral: ['D']


Une explication (1-1) impose une paire distincte (p,c) pour chaque c, donc il faut au moins autant de "pros" que de "cons" (disjonction côté pros).

In [39]:
print("|pros| =", len(pros), " , |cons| =", len(cons))

if len(pros) < len(cons):
    print("Conclusion : aucune explication (1-1) n'existe (pas assez de critères 'pour' pour couvrir tous les 'contre').")
else:
    print("Il y a assez de 'pros' en nombre : il faut encore vérifier la faisabilité avec les contraintes Δp+Δc>0.")


|pros| = 2  , |cons| = 4
Conclusion : aucune explication (1-1) n'existe (pas assez de critères 'pour' pour couvrir tous les 'contre').


Construction de tous les trade-offs (1-m) admissibles

In [40]:
def all_nonempty_subsets(lst):
    for r in range(1, len(lst)+1):
        for comb in combinations(lst, r):
            yield tuple(comb)

# Liste des trade-offs (1-m) admissibles : (p, S)
T_1m = []
for p in pros:
    for S in all_nonempty_subsets(cons):
        if delta[p] + sum(delta[c] for c in S) > 0:
            T_1m.append((p, S))

print("Nombre de trade-offs (1-m) admissibles =", len(T_1m))


Nombre de trade-offs (1-m) admissibles = 16


Modèle MILP (1-m) : couvrir tous les “cons” une seule fois et minimiser la longueur

In [41]:
m = Model("Explanation_1-m")

# Variables binaires : z[p,S] = 1 si on choisit le trade-off (p,S)
z = m.addVars(T_1m, vtype=GRB.BINARY, name="z")

# Chaque critère "contre" doit être couvert exactement une fois
for c in cons:
    m.addConstr(quicksum(z[p,S] for (p,S) in T_1m if c in S) == 1, name=f"cover_{c}")

# Chaque critère "pour" utilisé au plus une fois (trade-offs disjoints côté pros)
for p in pros:
    m.addConstr(quicksum(z[p,S] for (pp,S) in T_1m if pp == p) <= 1, name=f"use_{p}_at_most_once")

# Objectif : minimiser le nombre de trade-offs choisis
m.setObjective(quicksum(z[p,S] for (p,S) in T_1m), GRB.MINIMIZE)

m.params.OutputFlag = 0
m.optimize()


Résultat

In [42]:
status = m.Status

if status == GRB.OPTIMAL:
    E = [(p,S) for (p,S) in T_1m if z[p,S].X > 0.5]
    print("Explication (1-m) trouvée")
    print("Longueur l =", len(E))
    print("E =", E)

    print("\nInterprétation :")
    for p,S in E:
        total = delta[p] + sum(delta[c] for c in S)
        print(f"- {p} compense {list(S)} : {delta[p]} + {sum(delta[c] for c in S)} = {total} > 0")

elif status == GRB.INFEASIBLE:
    print("Aucune explication (1-m) n'existe : modèle infaisable.")
else:
    print("Statut solveur :", status)


Explication (1-m) trouvée
Longueur l = 1
E = [('A', ('B', 'C', 'E', 'F'))]

Interprétation :
- A compense ['B', 'C', 'E', 'F'] : 176 + -170 = 6 > 0
